# 3.9: Reproducibility

**Logging.** Record everything needed to rerun and interpret an experiment: the code version (a commit hash), the full configuration of hyperparameters, the data version, and training curves (loss, validation metrics, learning rate, gradient norm) over time. Experiment-tracking tools make this easy, but even a structured log file is far better than nothing.

**Checkpointing.** Save checkpoints periodically, including not only the model weights but also the **optimizer state** (Adam's moments), the **learning-rate scheduler state**, the step count, and the random number generator states. Resuming from weights alone restarts Adam's moment estimates from zero and resets the schedule, which can cause a loss spike and changes the run. For long runs, checkpoints are also insurance against hardware failures.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.

This snippet uses `step` from the surrounding training code. Those names are not defined in an earlier listing, so the cell shows the pattern as it appears in the chapter.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From Warmup plus decay for LLM pretraining
import math

def lr_at(step, peak_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    min_lr = peak_lr * min_lr_ratio
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(progress, 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

# PyTorch: apply via LambdaLR, which multiplies the optimizer's base lr
import torch

model = torch.nn.Linear(10, 10)

opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda step: lr_at(step, 1.0, warmup_steps=1000, total_steps=100_000))

# From Reproducibility
import random, numpy as np, torch


In [ ]:
torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
            "sched": sched.state_dict(), "step": step,
            "rng": torch.get_rng_state()}, "ckpt.pt")
